# NB08：全班能力試驗與數據品管（Proficiency Testing & Statistical Quality Control）

**食品分析實驗 週次 8｜Nielsen's Food Analysis, 6th ed. 對應章節：分析數據處理（Evaluating Analytical Data）、品質保證（Quality Assurance）**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 把全班 5 組共同盲樣（水分、灰分、蛋白質、脂肪）彙整成 proximate 表，並用差減法（by difference）算出碳水化合物，理解為什麼它的不確定度是其他四項的**平方和開根號**。
2. 用公式（而非查表）計算 Grubbs 檢定的臨界值，比較「組內 n=3」與「班級層級 n=5」兩種尺度下離群值檢定力的差異。
3. 用 scipy 與手算兩種方式做一元 ANOVA（one-way ANOVA），拆解重複性（$s_r$）與再現性（$s_R$）。
4. 計算 z-score（指定值＝中位數、$\sigma_{pt}$ 固定），並依 $|z|\le2$／$2<|z|<3$／$|z|\ge3$ 分級。
5. 畫 Shewhart 管制圖，並用程式自動偵測 5 條 Westgard 規則（$1_{3s}$、$2_{2s}$、$R_{4s}$、$4_{1s}$、$10_x$）。
6. 建立蛋白質（Kjeldahl）的量測不確定度預算，算出合併標準不確定度 $u_c$ 與擴充不確定度 $U$（$k=2$）。
7. 用「guard band（防護帶）」規則，對照結果 $\pm U$ 與規格，做出放行／不放行／無法判斷的決策。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**（與 W8 課堂 deck 完全相同的數字）。
- 若要換成你們班上實際彙整的數據，`load_data()` 函式支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb08_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，取得類似
     `https://docs.google.com/spreadsheets/d/e/xxx/pub?output=csv` 的網址，傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的上傳程式碼（使用 `google.colab.files.upload()`）。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，數字取自 W8 課堂 deck（`lab_decks/w08_proficiency/build_w08.py`），不是真實量測值，僅用於練習統計計算與繪圖。做作業時請換成你們班的實際彙整數據。

## 資料格式 (Data Schema) — 與其他週次筆記本共用

本課程所有週次的原始資料一律採用「長格式」（long format），欄位固定如下：

| 欄位 | 說明 |
|---|---|
| `group` | 組別代號。本週為 `G1`–`G5`（5 組共同盲樣），另有兩個「虛擬組別」：`QC`（管制圖的檢核樣品，跟哪一組做的無關）、`TA`（助教另外做的蛋白質重複性示意數據） |
| `student` | 本週與 `group` 相同：水分/灰分/蛋白質/脂肪是「整組共同一個結果」（3 位同學共用），不像 NB01 有逐生的重複測量 |
| `item` | 量測項目代號：`moisture`（水分）、`ash`（灰分）、`protein`（蛋白質，單次測定）、`fat`（脂肪，沿用 W6 盲樣直接 Soxhlet 三重複）、`fat_qc_check`（QC 檢核樣品）、`protein_repeatability`（助教蛋白質重複性） |
| `rep` | 重複次數（水分/灰分/脂肪為 1–3；蛋白質固定為 1，因為每組只有單次測定；`fat_qc_check` 為第幾次 run（1–12）；`protein_repeatability` 為助教第幾次重複（1–3）) |
| `value` | 量測值（本週皆為百分比，見 `unit`） |
| `unit` | `value` 的單位（本週固定為 `%`）|
| `temp_C` | 本週不適用（僅 NB01 的體積校正需要水溫），留空 |
| `note` | 備註欄，如：疑似離群值、系統性偏低、資料來源（W6 直接 Soxhlet）等（可空白）|

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`：一個可同時在 Colab 與本機執行、且不需要額外檔案就能跑的資料載入函式（內建模擬資料以字串形式嵌入本筆記本）。

**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb08_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
# Numbers reproduce lab_decks/w08_proficiency/build_w08.py exactly.
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note
G1,G1,moisture,1,2.75,%,,
G1,G1,moisture,2,2.78,%,,
G1,G1,moisture,3,2.8,%,,
G1,G1,ash,1,5.95,%,,
G1,G1,ash,2,6.02,%,,
G1,G1,ash,3,5.98,%,,
G1,G1,protein,1,25.93,%,,
G1,G1,fat,1,21.13,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G1,G1,fat,2,20.78,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G1,G1,fat,3,20.55,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G2,G2,moisture,1,2.85,%,,
G2,G2,moisture,2,2.88,%,,
G2,G2,moisture,3,2.83,%,,
G2,G2,ash,1,6.1,%,,
G2,G2,ash,2,6.05,%,,
G2,G2,ash,3,6.08,%,,
G2,G2,protein,1,26.17,%,,
G2,G2,fat,1,21.84,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G2,G2,fat,2,22.06,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G2,G2,fat,3,21.89,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G3,G3,moisture,1,2.7,%,,
G3,G3,moisture,2,2.74,%,,
G3,G3,moisture,3,3.3,%,,疑似離群值 (possible outlier)
G3,G3,ash,1,5.9,%,,
G3,G3,ash,2,5.95,%,,
G3,G3,ash,3,5.92,%,,
G3,G3,protein,1,26.03,%,,
G3,G3,fat,1,21.47,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G3,G3,fat,2,21.48,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G3,G3,fat,3,21.16,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G4,G4,moisture,1,2.9,%,,
G4,G4,moisture,2,2.92,%,,
G4,G4,moisture,3,2.89,%,,
G4,G4,ash,1,6.05,%,,
G4,G4,ash,2,6.0,%,,
G4,G4,ash,3,6.03,%,,
G4,G4,protein,1,24.47,%,,"系統性偏低 (systematic bias, ~1.5 個百分點)"
G4,G4,fat,1,21.19,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G4,G4,fat,2,21.16,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G4,G4,fat,3,21.52,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G5,G5,moisture,1,2.79,%,,
G5,G5,moisture,2,2.81,%,,
G5,G5,moisture,3,2.8,%,,
G5,G5,ash,1,5.98,%,,
G5,G5,ash,2,6.01,%,,
G5,G5,ash,3,5.99,%,,
G5,G5,protein,1,26.0,%,,
G5,G5,fat,1,21.14,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G5,G5,fat,2,21.27,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
G5,G5,fat,3,20.56,%,,W6 直接 Soxhlet(未水解) 盲樣三重複
QC,QC,fat_qc_check,1,25.95,%,,
QC,QC,fat_qc_check,2,26.15,%,,
QC,QC,fat_qc_check,3,25.85,%,,
QC,QC,fat_qc_check,4,26.2,%,,
QC,QC,fat_qc_check,5,25.9,%,,
QC,QC,fat_qc_check,6,26.05,%,,
QC,QC,fat_qc_check,7,26.1,%,,
QC,QC,fat_qc_check,8,26.95,%,,
QC,QC,fat_qc_check,9,25.8,%,,
QC,QC,fat_qc_check,10,25.3,%,,
QC,QC,fat_qc_check,11,25.25,%,,
QC,QC,fat_qc_check,12,26.0,%,,
TA,TA,protein_repeatability,1,25.95,%,,
TA,TA,protein_repeatability,2,26.15,%,,
TA,TA,protein_repeatability,3,25.85,%,,
"""

In [ ]:
def load_data(source=None):
    """Load NB08 proficiency-testing / QC data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb08_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb08_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['group'].nunique()} 個組別代號、"
      f"{df['item'].nunique()} 個量測項目。")
df.head()

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際彙整值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

In [ ]:
GROUPS = ["G1", "G2", "G3", "G4", "G5"]


def get_series(data, group, item):
    """Return the `value` column for one (group, item) as a numpy array,
    in `rep` order."""
    sub = data[(data["group"] == group) & (data["item"] == item)].sort_values("rep")
    return sub["value"].to_numpy(dtype=float)

## 2. Proximate 組成表與差減法碳水化合物 (Proximate Table & Carbohydrate by Difference)

把 5 組的水分、灰分、蛋白質、脂肪彙整成一張表（脂肪直接沿用 W6 的盲樣**直接 Soxhlet（未水解）**三重複）。**碳水化合物用差減法（by difference）計算**，不是直接測出來的：

$$\text{碳水化合物\%} = 100 - (\text{水分\%} + \text{灰分\%} + \text{蛋白質\%} + \text{脂肪\%})$$

差減法沒有自己的實驗步驟——它只是拿別人的結果做加減。因此水分、灰分、蛋白質、脂肪**各自的不確定度都會被帶進來**，而且要用「平方和開根號」合併（不確定度不能直接相加）：

$$u_{\text{碳水}} = \sqrt{u_{\text{水分}}^2 + u_{\text{灰分}}^2 + u_{\text{蛋白質}}^2 + u_{\text{脂肪}}^2}$$

蛋白質的標準不確定度 $u_{\text{蛋白質}}$ 需要一份完整的量測不確定度預算（詳見第 7 節），這裡先把它算出來備用（第 7 節會用同一組數字，做完整的分量拆解與說明）。

In [ ]:
moist_mean = {g: round(get_series(df, g, "moisture").mean(), 4) for g in GROUPS}
ash_mean = {g: round(get_series(df, g, "ash").mean(), 4) for g in GROUPS}
fat_mean = {g: round(get_series(df, g, "fat").mean(), 4) for g in GROUPS}
prot_mean = {g: round(get_series(df, g, "protein")[0], 4) for g in GROUPS}  # n=1，單次測定
carb_mean = {
    g: round(100.0 - (moist_mean[g] + ash_mean[g] + prot_mean[g] + fat_mean[g]), 4)
    for g in GROUPS
}

proximate_df = pd.DataFrame({
    "group": GROUPS,
    "moisture_%": [moist_mean[g] for g in GROUPS],
    "ash_%": [ash_mean[g] for g in GROUPS],
    "protein_%": [prot_mean[g] for g in GROUPS],
    "fat_%": [fat_mean[g] for g in GROUPS],
    "carbohydrate_%": [carb_mean[g] for g in GROUPS],
})
proximate_df

In [ ]:
# 蛋白質的「指定值」(assigned/reference value)：5 組單次測定值的中位數。
# 這個數字之後在第 5 節(z-score)會再用到一次(同一個數字，兩種用途)。
protein_reference_value = float(np.median([prot_mean[g] for g in GROUPS]))
print(f"蛋白質指定值 (5 組中位數) = {protein_reference_value:.4f}%")

# ---- 蛋白質的量測不確定度預算 (完整推導與說明在第 7 節；這裡先算出數值供 u_carb 使用) ----
u_weigh_rel = (0.0002 / np.sqrt(3)) / 1.0000 * 100          # 秤重: 分析天平 +-0.0002 g，矩形分布
u_titrant_rel = 0.30                                         # 滴定劑濃度: 標定後的標準不確定度(已給定)
u_vol_rel = (0.05 / np.sqrt(3)) / 25.0 * 100                # 滴定體積: 滴定管刻度 +-0.05 mL，矩形分布
u_factor_rel = (0.03 / np.sqrt(3)) / 6.38 * 100             # 轉換因子: 6.38 +- 0.03，矩形分布

ta_protein_reps = get_series(df, "TA", "protein_repeatability")
s_r_protein = float(np.std(ta_protein_reps, ddof=1))         # 重複性標準差(助教示範3重複)
u_repeat_rel = (s_r_protein / protein_reference_value) * 100  # 單次測定，不除以 sqrt(n)

u_components_rel = {
    "秤重 weighing": u_weigh_rel,
    "滴定劑濃度 titrant conc.": u_titrant_rel,
    "滴定體積 titration volume": u_vol_rel,
    "轉換因子 6.38": u_factor_rel,
    "重複性 repeatability": u_repeat_rel,
}
u_c_rel = float(np.sqrt(sum(v ** 2 for v in u_components_rel.values())))
u_c_abs = protein_reference_value * u_c_rel / 100.0
U_expanded = 2 * u_c_abs  # k=2

print(f"蛋白質 u_c(相對) = {u_c_rel:.4f}%   u_c(絕對) = {u_c_abs:.4f}%   U(k=2) = {U_expanded:.4f}%")

In [ ]:
# 其他三項的標準不確定度 (方法精密度，示意值，取自 W2/W3/W6 的重現性資料，此處直接給定)
u_moist_abs = 0.05
u_ash_abs = 0.08
u_fat_abs = 0.35   # Soxhlet 變異較大

u_carb_abs = float(np.sqrt(u_moist_abs ** 2 + u_ash_abs ** 2 + u_c_abs ** 2 + u_fat_abs ** 2))
U_carb_expanded = 2 * u_carb_abs

print("== 碳水化合物不確定度傳播 u_carb = sqrt(Sigma u^2) ==")
print(f"u_moist={u_moist_abs:.3f}  u_ash={u_ash_abs:.3f}  u_protein={u_c_abs:.3f}  u_fat={u_fat_abs:.3f}")
print(f"  ->  u_carb = {u_carb_abs:.4f}%   U_carb(k=2) = {U_carb_expanded:.4f}%")

可以看到，碳水化合物的不確定度（$u_{\text{碳水}}$）遠大於水分或灰分單獨的不確定度——這正是差減法「繼承所有人誤差」的代價，也是為什麼碳水化合物永遠是 proximate 分析中不確定度最大的一項。

**誤差不只傳遞「不確定度」，也會傳遞「偏差」**：表中的脂肪（約 21%）是 W6 的直接 Soxhlet（未水解），比乳品參考方法 **Röse–Gottlieb**（約 25.7%）低約 4.4 個百分點；差減法把這個低估**原封不動加到碳水化合物**，所以碳水被墊高到約 44%。$U_{\text{碳水}}$ 只涵蓋隨機的不確定度（約 ±0.8%），蓋不住這種方法偏差——第 9 節會用資料庫把它抓出來。

## 3. Grubbs 離群值檢定 (Grubbs' Test for Outliers)

Grubbs 統計量：

$$G = \frac{\max_i |x_i - \bar{x}|}{s}$$

雙側臨界值可以用 t 分布的公式直接算出來（不用查表）：

$$G_{crit} = \frac{n-1}{\sqrt{n}}\sqrt{\frac{t^2_{\alpha/(2n),\,n-2}}{n-2+t^2_{\alpha/(2n),\,n-2}}}$$

### 為什麼 n=3 時幾乎測不出離群值？

$G$ 有一個**理論上界**：不管離群值多誇張，$G$ 都不可能超過

$$G_{max} = \frac{n-1}{\sqrt{n}}$$

當 $n=3$ 時，$G_{max} = \frac{2}{\sqrt{3}} \approx 1.155$，而 $G_{crit}(n=3,\alpha=0.05)$ 剛好就卡在這條線附近——（$G_{crit} \approx 1.1543$，只比上界 1.1547 小 0.0004）代表**只有 3 個重複值時，這個檢定幾乎判不出顯著**：只有另外兩個值幾乎完全相同時，$G$ 才有機會超過臨界值，這也是為什麼水分/灰分/脂肪這種「組內三重複」的離群值檢定力很弱。

In [ ]:
def grubbs_crit(n, alpha=0.05):
    """兩側 Grubbs 臨界值:
    G_crit = (n-1)/sqrt(n) * sqrt(t^2/(n-2+t^2))，t = t.ppf(1-alpha/(2n), n-2)"""
    t = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    return (n - 1) / np.sqrt(n) * np.sqrt(t ** 2 / (n - 2 + t ** 2))


Gcrit3 = grubbs_crit(3)
Gcrit5 = grubbs_crit(5)
Gmax3 = 2 / np.sqrt(3)   # (n-1)/sqrt(n) at n=3, 理論上界
Gmax5 = 4 / np.sqrt(5)   # n=5 的理論上界

print(f"G_crit(n=3, a=.05) = {Gcrit3:.4f}   理論上界 (n-1)/sqrt(n) = {Gmax3:.4f}")
print(f"G_crit(n=5, a=.05) = {Gcrit5:.4f}   理論上界 (n-1)/sqrt(n) = {Gmax5:.4f}")

### 3.1 組內層級 (within-group, n=3)：G3 水分的三重複

In [ ]:
g3_moist = get_series(df, "G3", "moisture")
g3_mean, g3_sd = g3_moist.mean(), g3_moist.std(ddof=1)
g3_G = float(np.max(np.abs(g3_moist - g3_mean)) / g3_sd)

print(f"G3 水分三重複: {g3_moist}")
print(f"mean={g3_mean:.4f}  sd={g3_sd:.4f}  G={g3_G:.4f}  vs G_crit(n=3)={Gcrit3:.4f}  (理論上界={Gmax3:.4f})")
if g3_G < Gcrit3:
    print("=> G < G_crit，統計上『測不出來』——即使肉眼看 3.30% 很像離群值。")
else:
    print("=> G >= G_crit，判定為離群值。")

### 3.2 班級層級 (class-level, n=5)：5 組蛋白質單次測定值

In [ ]:
prot_arr = np.array([prot_mean[g] for g in GROUPS])
prot_grand_mean, prot_grand_sd = prot_arr.mean(), prot_arr.std(ddof=1)
prot_G = {g: float(abs(prot_mean[g] - prot_grand_mean) / prot_grand_sd) for g in GROUPS}
flagged_group = max(prot_G, key=prot_G.get)

grubbs_class_df = pd.DataFrame({
    "group": GROUPS,
    "protein_%": [prot_mean[g] for g in GROUPS],
    "G": [round(prot_G[g], 4) for g in GROUPS],
})
print(f"grand mean={prot_grand_mean:.4f}  sd={prot_grand_sd:.4f}")
print(grubbs_class_df)
print(f"\nflagged: {flagged_group}  G={prot_G[flagged_group]:.4f}  vs G_crit(n=5)={Gcrit5:.4f}")
if prot_G[flagged_group] > Gcrit5:
    print(f"=> {flagged_group} 的 G 超過臨界值，判定為離群值。")

## 4. 一元 ANOVA (One-Way ANOVA) — 以灰分為例

用一元變異數分析拆解「組內變異」與「組間變異」，並算出重複性 $s_r$、組間(實驗室間) $s_L$、再現性 $s_R$：

$$s_r = \sqrt{MSW} \qquad s_L = \sqrt{\max\!\big(0,\ \tfrac{MSB-MSW}{n}\big)} \qquad s_R = \sqrt{s_r^2 + s_L^2}$$

In [ ]:
ash_data = [get_series(df, g, "ash") for g in GROUPS]
F_scipy, p_scipy = stats.f_oneway(*ash_data)

k, n_rep = len(GROUPS), 3
N = k * n_rep
grand_mean_ash = float(np.mean([x for arr in ash_data for x in arr]))
SSB = n_rep * sum((arr.mean() - grand_mean_ash) ** 2 for arr in ash_data)
SSW = sum(((arr - arr.mean()) ** 2).sum() for arr in ash_data)
dfB, dfW = k - 1, N - k
MSB, MSW = SSB / dfB, SSW / dfW
F_manual = MSB / MSW
p_manual = float(stats.f.sf(F_manual, dfB, dfW))
s_r = float(np.sqrt(MSW))
s_L = float(np.sqrt(max(0.0, (MSB - MSW) / n_rep)))
s_R = float(np.sqrt(s_r ** 2 + s_L ** 2))
F_crit_ash = float(stats.f.ppf(0.95, dfB, dfW))

print("== ANOVA（灰分, k=5 組, n=3 重複）==")
print(f"手算: SSB={SSB:.5f}  SSW={SSW:.5f}  dfB={dfB}  dfW={dfW}  MSB={MSB:.5f}  MSW={MSW:.5f}")
print(f"F(手算)={F_manual:.4f}  F(scipy)={F_scipy:.4f}  p={p_manual:.4f}  F_crit(0.05)={F_crit_ash:.4f}")
print(f"s_r(重複性)={s_r:.4f}  s_L(組間)={s_L:.4f}  s_R(再現性)={s_R:.4f}")

F 遠大於臨界值、p < 0.05：5 組灰分平均值之間的差異，不能只用組內的重複性（$s_r$）來解釋——代表組間存在真實的系統性落差。這正是為什麼再現性 $s_R$ 一定要考慮組間變異，不能只看自己組的重複測定。

## 5. z-score 與分級 (z-scores & Classification)

$$z = \frac{x_{\text{組}} - x_{\text{assigned}}}{\sigma_{pt}}$$

- $x_{\text{assigned}}$：指定值，用**中位數**（穩健統計量，不容易被離群組拉走）。
- $\sigma_{pt}$：事先訂定的目標標準差（fitness-for-purpose，本課採**固定值** 0.40%，不是用受汙染的班級 SD）。
- 分級規則：$|z|\le2$ 滿意（satisfactory）；$2<|z|<3$ 問題（questionable）；$|z|\ge3$ 不滿意（unsatisfactory）。

In [ ]:
z_assigned = protein_reference_value   # 與第 2 節算出的指定值是同一個數字
sigma_pt = 0.40
z_score = {g: round((prot_mean[g] - z_assigned) / sigma_pt, 3) for g in GROUPS}


def classify_z(z):
    if abs(z) <= 2:
        return "satisfactory"
    if abs(z) < 3:
        return "questionable"
    return "unsatisfactory"


z_df = pd.DataFrame({
    "group": GROUPS,
    "protein_%": [prot_mean[g] for g in GROUPS],
    "z": [z_score[g] for g in GROUPS],
    "category": [classify_z(z_score[g]) for g in GROUPS],
})
print(f"x_assigned = {z_assigned:.4f}%   sigma_pt = {sigma_pt:.2f}%")
z_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:red" if abs(z_score[g]) >= 3 else ("tab:orange" if abs(z_score[g]) > 2 else "tab:green")
          for g in GROUPS]
ax.bar(GROUPS, [z_score[g] for g in GROUPS], color=colors)
for lim, style in [(2, "--"), (-2, "--"), (3, ":"), (-3, ":")]:
    ax.axhline(lim, color="grey", linestyle=style, linewidth=1)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("z-score (protein)")
ax.set_title("Protein z-scores by group (dashed = +-2, dotted = +-3)")
plt.tight_layout()
plt.show()

## 6. Shewhart 管制圖與 Westgard 規則 (Shewhart Control Chart & Westgard Rules)

QC 檢核樣品的 12 次分析 run。**管制界線用「既有基準」**（歷史建立的 mean / s，不是從這 12 點自己算出來的——否則違規點會被界線自己吃掉）。

本節用程式自動偵測 5 條常見的 Westgard 規則：

- **$1_{3s}$**：任一點超出 $\pm3s$。
- **$2_{2s}$**：連續兩點同側超出 $\pm2s$。
- **$R_{4s}$**：相鄰兩點的全距（range）超過 $4s$（例如一點在 $+2s$ 以上，下一點在 $-2s$ 以下）。
- **$4_{1s}$**：連續四點同側超出 $\pm1s$。
- **$10_x$**：連續十點落在中心線同一側（不論距離多遠）。

In [ ]:
qc_mean, qc_s = 26.0, 0.30   # 既有基準(歷史值)，非由下面 12 點自算
qc_runs = get_series(df, "QC", "fat_qc_check")
qc_z = [(v - qc_mean) / qc_s for v in qc_runs]

print(f"既有基準: mean={qc_mean:.2f}%  s={qc_s:.2f}%")
for i, (v, z) in enumerate(zip(qc_runs, qc_z), start=1):
    print(f"  Run {i:2d}: value={v:.2f}%  z={z:+.3f}")

In [ ]:
def rule_1_3s(z):
    return [i for i, zi in enumerate(z) if abs(zi) > 3]


def rule_2_2s(z):
    viol = set()
    for i in range(len(z) - 1):
        if abs(z[i]) > 2 and abs(z[i + 1]) > 2 and (z[i] > 0) == (z[i + 1] > 0):
            viol.update([i, i + 1])
    return sorted(viol)


def rule_R_4s(z):
    return [i for i in range(len(z) - 1) if abs(z[i] - z[i + 1]) > 4]


def rule_4_1s(z):
    viol = set()
    for i in range(len(z) - 3):
        window = z[i:i + 4]
        if all(abs(w) > 1 for w in window) and (all(w > 0 for w in window) or all(w < 0 for w in window)):
            viol.update(range(i, i + 4))
    return sorted(viol)


def rule_10_x(z):
    viol = set()
    for i in range(len(z) - 9):
        window = z[i:i + 10]
        if all(w > 0 for w in window) or all(w < 0 for w in window):
            viol.update(range(i, i + 10))
    return sorted(viol)


westgard_results = {
    "1_3s": rule_1_3s(qc_z),
    "2_2s": rule_2_2s(qc_z),
    "R_4s": rule_R_4s(qc_z),
    "4_1s": rule_4_1s(qc_z),
    "10_x": rule_10_x(qc_z),
}

print("== Westgard 規則自動偵測結果（違規點的 run 編號，1-based）==")
for rule, idx in westgard_results.items():
    runs = [i + 1 for i in idx]
    status = f"違規於 run {runs}" if runs else "沒有違規"
    print(f"  {rule:5s}: {status}")

在這組 QC 資料中，只有 **$1_{3s}$**（第 8 次 run，突然衝到 26.95%）與 **$2_{2s}$**（第 10、11 次連續兩點都低於 $-2s$）被觸發；$R_{4s}$、$4_{1s}$、$10_x$ 都沒有被觸發。這是刻意設計的教學重點：**不同的 Westgard 規則對「不同型態的異常」敏感度不同**——$1_{3s}$ 抓的是單次劇烈失控，$2_{2s}$ 抓的是短期系統性偏移，而 $4_{1s}$/$10_x$ 是用來抓「緩慢飄移」的規則，這批資料裡沒有那種緩慢趨勢，所以不會觸發。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
runs_x = np.arange(1, len(qc_runs) + 1)
all_viol = sorted(set(i for idx in westgard_results.values() for i in idx))
point_colors = ["tab:red" if i in all_viol else "tab:blue" for i in range(len(qc_runs))]

ax.plot(runs_x, qc_runs, color="tab:blue", linewidth=1.5, zorder=2)
ax.scatter(runs_x, qc_runs, c=point_colors, s=[70 if i in all_viol else 35 for i in range(len(qc_runs))], zorder=3)

for k_, style, color in [(0, "-", "black"), (2, "--", "tab:orange"), (3, ":", "tab:red")]:
    if k_ == 0:
        ax.axhline(qc_mean, color=color, linestyle=style, linewidth=1.2, label="mean")
    else:
        ax.axhline(qc_mean + k_ * qc_s, color=color, linestyle=style, linewidth=1.2, label=f"+/-{k_}s")
        ax.axhline(qc_mean - k_ * qc_s, color=color, linestyle=style, linewidth=1.2)

ax.set_xlabel("Run number")
ax.set_ylabel("QC value (%)")
ax.set_title("Shewhart control chart: QC check sample (established mean/s)")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 7. 蛋白質量測不確定度預算 (Measurement Uncertainty Budget — Protein)

這裡把第 2 節已經算過的數字重新攤開，逐一說明每個分量的來源。Kjeldahl 蛋白質測定的不確定度來源：

| 來源 | 說明 |
|---|---|
| 秤重 (weighing) | 分析天平 $\pm0.0002$ g，矩形分布，標準不確定度 $= \dfrac{\text{半寬}}{\sqrt3}$ |
| 滴定劑濃度 (titrant conc.) | 0.1 N HCl 標定後的標準不確定度（已經是標準差型式，直接使用） |
| 滴定體積 (titration volume) | 50 mL 滴定管刻度 $\pm0.05$ mL，矩形分布 |
| 轉換因子 (conversion factor) | 牛乳蛋白質係數 $6.38\pm0.03$，矩形分布 |
| 重複性 (repeatability) | 每組蛋白質**只有單次測定**（n=1，見 W5 消化爐 8 管限制），無法用組內重複反推 $s_r$；改用助教另外對同一份奶粉做的 3 重複測定，得到方法的 $s_r$。**因為每組報告的就是這一個單次值（未再平均），重複性直接以 $s_r$ 貢獻，不再除以 $\sqrt{n}$**。 |

矩形分布（儀器刻度/公差）的標準不確定度 = 半寬 ÷ $\sqrt3$；已經是標準差型式的分量（如滴定劑標定）則直接使用。合併與擴充：

$$u_c = \sqrt{\sum (\text{各分量})^2} \qquad U = k \cdot u_c \ \ (k=2,\ \approx95\%\text{信心水準})$$

In [ ]:
budget_df = pd.DataFrame({
    "component": list(u_components_rel.keys()),
    "relative_std_uncertainty_%": [round(v, 4) for v in u_components_rel.values()],
})
print(f"助教 3 重複測定值: {ta_protein_reps}  ->  s_r(重複性) = {s_r_protein:.4f}%")
budget_df

In [ ]:
print(f"合併相對標準不確定度 u_c(rel) = {u_c_rel:.4f}%")
print(f"指定值 (reference value)      = {protein_reference_value:.4f}%")
print(f"合併絕對標準不確定度 u_c(abs) = {u_c_abs:.4f}%")
print(f"擴充不確定度 U (k=2)          = {U_expanded:.4f}%")
print()
print(f"=> 報告寫法：蛋白質 = {protein_reference_value:.2f}% +/- {U_expanded:.2f}%  (U, k=2)")

## 8. 放行判定：Guard Band 規則 (Release Decision with Guard Band)

回到開場的問題：奶油測得 **80.3% ± 0.4%**（U, k=2），規格是乳脂 $\ge80\%$，這批**可以放行**嗎？

**guard band（防護帶）規則**（本課採用）：

- 規格是「$\ge$ 下限」：只有當**最壞情況（結果 $-U$）$\ge$ 規格**才判定**放行**；當**最佳情況（結果 $+U$）$<$ 規格**時判定**不放行**；其餘（規格落在結果$\pm U$ 區間內）判定**無法判斷**。
- 規格是「$\le$ 上限」：只有當**最壞情況（結果 $+U$）$\le$ 規格**才判定**放行**；當**最佳情況（結果 $-U$）$>$ 規格**時判定**不放行**；其餘判定**無法判斷**。

「測到的數字比規格高（或低）」跟「這批貨真的合格」不是同一件事——不確定度存在的意義，就是讓「合格」不再只靠一個點估計數字說了算。

In [ ]:
def release_decision(result, U, spec, spec_type):
    """spec_type: ">=" (至少要達到規格，如乳脂/蛋白質下限)
                  "<=" (不能超過規格，如水分/鈉上限)"""
    if spec_type == ">=":
        worst, best = result - U, result + U
        if worst >= spec:
            return "放行 (pass)", worst
        if best < spec:
            return "不放行 (fail)", worst
        return "無法判斷 (inconclusive)", worst
    elif spec_type == "<=":
        worst, best = result + U, result - U
        if worst <= spec:
            return "放行 (pass)", worst
        if best > spec:
            return "不放行 (fail)", worst
        return "無法判斷 (inconclusive)", worst
    raise ValueError("spec_type must be '>=' or '<='")


release_cases = pd.DataFrame([
    {"case": "奶油 乳脂 (butter fat)", "result": 80.3, "U": 0.4, "spec": 80.0, "spec_type": ">="},
    {"case": "全脂奶粉 蛋白質 (milk powder protein)", "result": 26.5, "U": 0.3, "spec": 25.0, "spec_type": ">="},
    {"case": "全脂奶粉 水分 (milk powder moisture)", "result": 3.6, "U": 0.15, "spec": 3.5, "spec_type": "<="},
    {"case": "罐頭湯 鈉 (canned soup sodium, mg/100g)", "result": 620, "U": 15, "spec": 500, "spec_type": "<="},
    {"case": "果汁 維生素C (juice vitamin C, mg/100mL)", "result": 33.0, "U": 1.0, "spec": 30.0, "spec_type": ">="},
    {"case": "巧克力 水分 (chocolate moisture)", "result": 2.30, "U": 0.10, "spec": 2.0, "spec_type": "<="},
])

decisions, worst_cases = [], []
for _, r in release_cases.iterrows():
    decision, worst = release_decision(r["result"], r["U"], r["spec"], r["spec_type"])
    decisions.append(decision)
    worst_cases.append(round(worst, 4))
release_cases["worst_case"] = worst_cases
release_cases["decision"] = decisions
release_cases

特別留意第一列（奶油）：點估計 80.3% 已經高於規格 80%，但最壞情況 $80.3-0.4=79.9\%$ 跌破規格，所以判定是**無法判斷**，不是「放行」——即使測到的數字看起來合格。

## 9. 用資料庫檢查全班盲樣：全脂奶粉 vs USDA 與 TFND (Food-composition database check)

前面的 z-score、Grubbs、ANOVA 都是「拿全班互相比」。這一節換一個**外部**參考點：官方食物成分資料庫中的全脂奶粉——
**USDA FoodData Central SR Legacy 170876**（*Milk, dry, whole, with added vitamin D*；不添加維生素 D 的 173454 一般成分數值完全相同）與**衛福部食藥署 食品營養成分資料庫 L0200101「全脂奶粉」**。

要回答兩個問題：

1. 全班的盲樣結果，對「全脂奶粉」來說合理嗎？
2. 如果不合理，是「**我這組**有問題」，還是「**全班／這個方法**有問題」？

**資料來源與授權（使用本節數字時請一併註明）**

- **USDA FoodData Central**：SR Legacy（2018-04 最終版）與 Foundation Foods（2026-04-30 版），美國農業部農業研究署（USDA ARS），公共領域／CC0 1.0。<https://fdc.nal.usda.gov>
- **衛生福利部食品藥物管理署「食品營養成分資料庫」**（政府資料開放平臺 [data.gov.tw 資料集 8543](https://data.gov.tw/dataset/8543)「食品營養成分資料集」），依**政府資料開放授權條款第1版**釋出，使用時須註明出處。
- 整理後的檔案：`data/reference_foods.json`（每筆都有資料庫、版本、品項 id、樣本數 n、SD 或 min/max）。由教師端的 `build_reference_foods.py` 直接從官方檔案產生，**沒有任何一個數字是手打或教學假設值**；唯一的教學假設是下面說明的「預設容許差」。

**合理範圍怎麼來的（教學假設要講清楚）**

1. 每一筆資料庫數值先算自己的區間：有 **min–max**（USDA 部分品項）就用 min–max；有 **SD 且 n ≥ 2**（TFND）就用 **平均 ± 2 SD**。
2. 資料庫只代表「被分析的那幾個產品」，不同品牌/批次之間的差異通常更大，所以每筆區間**至少**保留「教學預設容許差」：數值 ± max(相對容許差, 絕對下限)——水分/蛋白質/碳水 ±10%、脂肪/灰分/總糖/鈉 ±15%（絕對下限：水分/蛋白質/脂肪 0.3 g、灰分 0.05 g、碳水 1 g、總糖 0.5 g、鈉 5 mg，皆為每 100 g）。**這是教學假設，不是資料庫數值。**
3. 把各筆區間聯集起來＝「合理範圍」。`within_entries` 欄位另外列出**每一個**資料庫品項是否涵蓋你的數值，避免「只被其中一個資料庫撐住的範圍內」。

In [ ]:
# ---- 內嵌的參考資料子集（離線備援；由 data/reference_foods.json 自動產生，勿手改）----
# 來源與授權見 meta.sources：USDA FoodData Central (CC0) 與 衛福部食藥署 食品營養成分資料庫
# (政府資料開放授權條款第1版，資料集 data.gov.tw 8543)。
REFERENCE_FALLBACK_JSON = r"""{"meta":{"built":"2026-09-27","units":"per 100 g edible portion (sodium in mg, everything else in g)","sources":[{"db":"USDA FDC","dataset":"SR Legacy","release":"2018-04 (final SR release; FDC CSV 2018-04)","url":"https://fdc.nal.usda.gov/download-datasets","file":"FoodData_Central_sr_legacy_food_csv_2018-04.zip","retrieved_via":"bulk CSV FoodData_Central_sr_legacy_food_csv_2018-04.zip","licence":"Public domain / CC0 1.0 (USDA ARS). Suggested citation: U.S. Department of Agriculture, Agricultural Research Service. FoodData Central, SR Legacy (2018). fdc.nal.usda.gov"},{"db":"USDA FDC","dataset":"Foundation Foods","release":"2026-04-30","url":"https://fdc.nal.usda.gov/download-datasets","file":"FoodData_Central_foundation_food_csv_2026-04-30.zip","retrieved_via":"bulk CSV FoodData_Central_foundation_food_csv_2026-04-30.zip","licence":"Public domain / CC0 1.0 (USDA ARS). Suggested citation: U.S. Department of Agriculture, Agricultural Research Service. FoodData Central, Foundation Foods (2026-04-30). fdc.nal.usda.gov"},{"db":"TFND","dataset":"食品營養成分資料集 (Taiwan Food Nutrient Database, open-data export)","release":"data.gov.tw metadata updated 2026-08-27 14:08; update frequency 每3月","url":"https://data.gov.tw/dataset/8543","file":"https://data.fda.gov.tw/data/opendata/export/20/csv (zip, UTF-8 CSV)","publisher":"衛生福利部食品藥物管理署 (Taiwan FDA)","licence":"政府資料開放授權條款-第1版 (Open Government Data License, version 1.0); attribution required","attribution":"資料來源：衛生福利部食品藥物管理署「食品營養成分資料庫」，政府資料開放平臺 data.gov.tw 資料集 8543，依政府資料開放授權條款第1版釋出。"}],"nutrients":{"water":{"unit":"g","usda_nutrient_id":[1051],"tfnd_item":"水分"},"protein":{"unit":"g","usda_nutrient_id":[1003],"tfnd_item":"粗蛋白"},"fat":{"unit":"g","usda_nutrient_id":[1004],"tfnd_item":"粗脂肪"},"ash":{"unit":"g","usda_nutrient_id":[1007],"tfnd_item":"灰分"},"carbohydrate":{"unit":"g","usda_nutrient_id":[1005],"tfnd_item":"總碳水化合物"},"sugars_total":{"unit":"g","usda_nutrient_id":[2000,1063],"tfnd_item":"糖質總量"},"sodium":{"unit":"mg","usda_nutrient_id":[1093],"tfnd_item":"鈉"}},"fields":{"n":"USDA data_points / TFND 樣本數","sd":"TFND 標準差 only","min/max/median":"USDA only, when reported","derivation":"USDA derivation code: description","n_factor":"nitrogen-to-protein factor (USDA food_protein_conversion_factor); null = not given by the database"},"notes":["All values are copied from the official releases by build_reference_foods.py; no value is typed by hand.","TFND 粗蛋白 open data does not state the nitrogen-to-protein factor, so n_factor is null for TFND entries.","USDA SR Legacy 'Sugars, Total' can be derived from other nutrients (see derivation) rather than measured.","'match' = exact | closest; read match_note and the food's variability_note before interpreting a comparison."],"subset_of":"data/reference_foods.json","subset_keys":["whole_milk_powder","nonfat_milk_powder"]},"foods":[{"key":"whole_milk_powder","name_zh":"全脂奶粉","name_en":"Whole milk powder","tags":["dairy_powder"],"variability_note":"奶粉脂肪/蛋白質隨品牌與標準化程度而異；課程盲樣是單一品牌，不一定等於資料庫平均。","entries":[{"db":"USDA","dataset":"SR Legacy","id":170876,"name":"Milk, dry, whole, with added vitamin D","n_factor":6.38,"nutrients":{"water":{"value":2.47,"unit":"g","n":25,"nutrient_id":1051},"protein":{"value":26.32,"unit":"g","n":22,"nutrient_id":1003},"fat":{"value":26.71,"unit":"g","n":30,"nutrient_id":1004},"ash":{"value":6.08,"unit":"g","n":20,"nutrient_id":1007},"carbohydrate":{"value":38.42,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":38.42,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sodium":{"value":371.0,"unit":"mg","n":7,"nutrient_id":1093}},"match":"exact","match_note":"課程指定主要參考值。注意：170876 的正式名稱是 'with added vitamin D'；'without added vitamin D' 是 173454，兩者一般成分數值相同。"},{"db":"USDA","dataset":"SR Legacy","id":173454,"name":"Milk, dry, whole, without added vitamin D","n_factor":6.38,"nutrients":{"water":{"value":2.47,"unit":"g","n":25,"nutrient_id":1051},"protein":{"value":26.32,"unit":"g","n":22,"nutrient_id":1003},"fat":{"value":26.71,"unit":"g","n":30,"nutrient_id":1004},"ash":{"value":6.08,"unit":"g","n":20,"nutrient_id":1007},"carbohydrate":{"value":38.42,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":38.42,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sodium":{"value":371.0,"unit":"mg","n":7,"nutrient_id":1093}},"match":"exact","match_note":"170876 的不添加維生素 D 版本（一般成分相同）。"},{"db":"TFND","dataset":"食品營養成分資料集","id":"L0200101","name":"全脂奶粉","name_en":"Whole milk powder ","description":"樣品狀態:全脂(26-42%)奶粉; 前處理描述:混合均勻","category":"乳品類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":2.8,"unit":"g","n":2,"sd":0.1,"tfnd_item":"水分"},"protein":{"value":26.4,"unit":"g","n":2,"sd":0.3,"tfnd_item":"粗蛋白"},"fat":{"value":28.2,"unit":"g","n":2,"sd":0.6,"tfnd_item":"粗脂肪"},"ash":{"value":5.6,"unit":"g","n":2,"sd":0.0,"tfnd_item":"灰分"},"carbohydrate":{"value":37.0,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":36.4,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":337.0,"unit":"mg","n":2,"sd":70.0,"tfnd_item":"鈉"}},"match":"exact"}]},{"key":"nonfat_milk_powder","name_zh":"脫脂奶粉","name_en":"Nonfat (skim) milk powder","tags":["dairy_powder"],"variability_note":"即溶/一般型、是否強化會影響數值。","entries":[{"db":"USDA","dataset":"SR Legacy","id":170877,"name":"Milk, dry, nonfat, regular, without added vitamin A and vitamin D","n_factor":6.38,"nutrients":{"water":{"value":3.16,"unit":"g","n":382,"nutrient_id":1051},"protein":{"value":36.16,"unit":"g","n":183,"nutrient_id":1003},"fat":{"value":0.77,"unit":"g","n":351,"nutrient_id":1004},"ash":{"value":7.93,"unit":"g","n":104,"nutrient_id":1007},"carbohydrate":{"value":51.98,"unit":"g","n":0,"nutrient_id":1005,"derivation":"NC: Calculated"},"sugars_total":{"value":51.98,"unit":"g","n":0,"nutrient_id":2000,"derivation":"NR: Nutrient that is based on other nutrient/s; value used directly, ex. Nut.#204 from Nut.#298"},"sodium":{"value":535.0,"unit":"mg","n":6,"nutrient_id":1093}},"match":"exact","match_note":"regular, without added vitamin A and D"},{"db":"TFND","dataset":"食品營養成分資料集","id":"L0207101","name":"脫脂奶粉","name_en":"Skimmed milk powder ","description":"樣品狀態:脫脂(<1.5%)奶粉; 前處理描述:混合均勻","category":"乳品類","record_type":"樣品基本資料","n_factor":null,"nutrients":{"water":{"value":4.0,"unit":"g","n":1,"tfnd_item":"水分"},"protein":{"value":36.2,"unit":"g","n":1,"tfnd_item":"粗蛋白"},"fat":{"value":0.9,"unit":"g","n":1,"tfnd_item":"粗脂肪"},"ash":{"value":7.7,"unit":"g","n":1,"tfnd_item":"灰分"},"carbohydrate":{"value":51.2,"unit":"g","tfnd_item":"總碳水化合物"},"sugars_total":{"value":36.6,"unit":"g","tfnd_item":"糖質總量"},"sodium":{"value":338.0,"unit":"mg","n":1,"tfnd_item":"鈉"}},"match":"exact"}]}]}"""

In [ ]:
# >>> DBCHECK-HELPERS-BEGIN
# ---- 資料庫比對小工具 (food-composition database check helpers) ------------
# 同一份程式碼嵌在 NB03 與 NB08；原始檔：colab_notebooks/_dbcheck_common.py
import io as _db_io
import json as _db_json
import math as _db_math
import os as _db_os
import urllib.request as _db_urlreq

import numpy as _db_np
import pandas as _db_pd
import matplotlib.pyplot as _db_plt

DB_REFERENCE_URL = ("https://raw.githubusercontent.com/Tai-ShengYeh/"
                    "food-analysis-lab/main/data/reference_foods.json")

# 教學假設 (teaching assumption)：資料庫只給單一平均值、沒有 min/max 或 SD 時，
# 用「相對容許差 rel」與「絕對下限 abs」取較大者，當作『不同產品之間』合理的差異範圍。
# 有 min/max 或 SD 時，也至少保留這個寬度（n=2 的 SD 或 SD=0 會嚴重低估產品間差異）。
DB_DEFAULT_TOL = {
    #  nutrient      (rel,  abs)   abs 的單位：g/100 g（鈉為 mg/100 g）
    "water":        (0.10, 0.3),
    "protein":      (0.10, 0.3),
    "fat":          (0.15, 0.3),
    "ash":          (0.15, 0.05),
    "carbohydrate": (0.10, 1.0),
    "sugars_total": (0.15, 0.5),
    "sodium":       (0.15, 5.0),
}
DB_UNITS = {"water": "g", "protein": "g", "fat": "g", "ash": "g",
            "carbohydrate": "g", "sugars_total": "g", "sodium": "mg"}
DB_NACL_FACTOR = 58.44 / 22.99  # 2.542


def load_reference(source=None, url=DB_REFERENCE_URL, timeout=5, fallback_json=None):
    """Load reference_foods.json. Order: `source` (path or URL) -> local
    data/reference_foods.json -> GitHub raw URL -> embedded fallback subset
    (REFERENCE_FALLBACK_JSON, so the notebook always runs offline).
    The source actually used is stored in ref["_loaded_from"].
    Set environment variable DBCHECK_OFFLINE=1 to skip the network."""
    offline = _db_os.environ.get("DBCHECK_OFFLINE", "") == "1"
    tries = []
    if source is not None:
        tries.append(("source", source))
    tries.append(("local", _db_os.path.join("data", "reference_foods.json")))
    if not offline and url:
        tries.append(("url", url))
    for kind, where in tries:
        try:
            if str(where).startswith(("http://", "https://")):
                if offline:
                    continue
                with _db_urlreq.urlopen(str(where), timeout=timeout) as r:
                    ref = _db_json.loads(r.read().decode("utf-8"))
            elif _db_os.path.exists(str(where)):
                with open(str(where), encoding="utf-8") as fh:
                    ref = _db_json.load(fh)
            else:
                continue
            ref["_loaded_from"] = f"{kind}: {where}"
            print(f"參考資料庫來源：{ref['_loaded_from']}（{len(ref['foods'])} 種食品）")
            return ref
        except Exception as e:  # noqa: BLE001
            print(f"讀取 {where} 失敗（{type(e).__name__}），改試下一個來源。")
    text = fallback_json if fallback_json is not None else globals().get("REFERENCE_FALLBACK_JSON")
    if text is None:
        raise RuntimeError("找不到 reference_foods.json，也沒有內嵌的備援資料。")
    ref = _db_json.loads(text)
    ref["_loaded_from"] = "embedded"
    print(f"參考資料庫來源：embedded（筆記本內嵌子集，{len(ref['foods'])} 種食品）")
    return ref


def db_food(ref, food_key):
    for f in ref["foods"]:
        if f["key"] == food_key:
            return f
    raise KeyError(f"reference 裡沒有 food_key={food_key!r}；可用：{[f['key'] for f in ref['foods']]}")


def _db_fmt(x, nd=4):
    if x is None or (isinstance(x, float) and _db_math.isnan(x)):
        return "NA"
    return f"{x:.{nd}g}" if abs(x) < 1000 else f"{x:.0f}"


def _db_entry_tag(e):
    ds = {"SR Legacy": "SR", "Foundation": "FF"}.get(e.get("dataset"), "")
    return f"{e['db']}{'-' + ds if ds else ''} {e['id']}"


def reference_range(ref, food_key, nutrient, verbose=False):
    """Combine all database entries for one food/nutrient into a plausible range.
    Each entry -> interval = union of (a) its statistics [min,max] or mean±2SD (n>=2)
    and (b) the teaching default tolerance value±max(rel·value, abs).
    The food's range = union of the entry intervals.  Returns a dict with
    low, high, center (mean of entry values), water center, and per-entry details."""
    food = db_food(ref, food_key)
    rel, absf = DB_DEFAULT_TOL[nutrient]
    rows = []
    for e in food["entries"]:
        v = e["nutrients"].get(nutrient)
        if v is None or v.get("value") is None:
            continue
        val = float(v["value"])
        half = max(rel * abs(val), absf)
        lo, hi = val - half, val + half
        rule = f"±max({rel:.0%}, {absf:g}) 教學預設"
        if v.get("min") is not None and v.get("max") is not None:
            lo, hi = min(lo, v["min"]), max(hi, v["max"])
            rule = f"min–max {_db_fmt(v['min'])}–{_db_fmt(v['max'])} ∪ " + rule
        elif v.get("sd") is not None and (v.get("n") or 0) >= 2:
            lo, hi = min(lo, val - 2 * v["sd"]), max(hi, val + 2 * v["sd"])
            rule = f"mean±2SD (SD={_db_fmt(v['sd'])}, n={v['n']}) ∪ " + rule
        lo = max(lo, 0.0)
        rows.append({"entry": _db_entry_tag(e), "db": e["db"], "dataset": e.get("dataset"),
                     "id": e["id"], "name": e["name"], "match": e.get("match"),
                     "value": val, "n": v.get("n"), "sd": v.get("sd"),
                     "min": v.get("min"), "max": v.get("max"),
                     "derivation": v.get("derivation"), "n_factor": e.get("n_factor"),
                     "low": lo, "high": hi, "rule": rule})
    if not rows:
        raise ValueError(f"{food_key} 沒有任何資料庫提供 {nutrient}")
    water = [e["nutrients"]["water"]["value"] for e in food["entries"]
             if e["nutrients"].get("water", {}).get("value") is not None]
    out = {"food_key": food_key, "nutrient": nutrient, "unit": DB_UNITS[nutrient],
           "low": min(r["low"] for r in rows), "high": max(r["high"] for r in rows),
           "center": float(_db_np.mean([r["value"] for r in rows])),
           "water_center": float(_db_np.mean(water)) if water else None,
           "entries": rows, "food": food}
    if verbose:
        print(f"{food['name_zh']} ({food_key}) — {nutrient}, {DB_UNITS[nutrient]}/100 g")
        for r in rows:
            print(f"  {r['entry']:<16} {r['name'][:48]:<48} value={_db_fmt(r['value'])}"
                  f"  n={r['n']}  -> {_db_fmt(r['low'])}–{_db_fmt(r['high'])}  [{r['rule']}]")
        print(f"  => 合理範圍 (union) = {_db_fmt(out['low'])}–{_db_fmt(out['high'])} {DB_UNITS[nutrient]}/100 g")
    return out


def _db_to_per100g(nutrient, value, unit, density=None):
    """Convert a reported value to the database basis (g or mg per 100 g). Returns (value, note)."""
    u = (unit or "").replace(" ", "").lower()
    if nutrient == "sodium":
        if u in ("", "mg/100g", "mg"):
            return value, None
        if u in ("mg/kg", "ppm", "μg/g", "ug/g"):
            return value / 10.0, "mg/kg (ppm) ÷ 10 → mg/100 g"
        if u in ("g/100g",):
            return value * 1000.0, "g/100 g × 1000 → mg/100 g"
        if u in ("mg/100ml",):
            d = density if density else 1.0
            note = (f"mg/100 mL ÷ 密度 {d:g} g/mL → mg/100 g" if density
                    else "mg/100 mL 當作 mg/100 g（未提供密度，假設 1 g/mL）")
            return value / d, note
        if u in ("mg/l",):
            d = density if density else 1.0
            return value / 10.0 / d, f"mg/L ÷ 10 ÷ 密度 {d:g} → mg/100 g"
        raise ValueError(f"不認得的鈉單位：{unit}")
    if u in ("", "%", "g/100g", "g", "%w/w"):
        return value, None
    if u in ("g/kg",):
        return value / 10.0, "g/kg ÷ 10 → g/100 g"
    if u in ("g/100ml",):
        d = density if density else 1.0
        return value / d, f"g/100 mL ÷ 密度 {d:g} → g/100 g"
    raise ValueError(f"不認得的單位：{unit}")


def check_measurement(ref, food_key, nutrient, value, unit=None, basis="wet", method=None,
                      n_factor=None, sd=None, n=None, dilution=None, density=None,
                      water=None, label=None):
    """Compare one (mean) measurement with the reference range and run diagnostic flags.
    basis: "wet" (as is) or "dry" (reported on dry matter; converted with `water`
    or the database water).  method: free text, e.g. "soxhlet", "rose_gottlieb",
    "kjeldahl", "ise", "phenol_sulfuric", "difference".  n_factor: the N-to-protein
    factor YOU used.  dilution: the dilution factor you used (to detect a forgotten one)."""
    rr = reference_range(ref, food_key, nutrient)
    food = rr["food"]
    flags, notes = [], []
    v, conv = _db_to_per100g(nutrient, float(value), unit, density)
    if conv:
        notes.append("單位換算：" + conv)
    if basis == "dry":
        w = water if water is not None else rr["water_center"]
        if w is not None:
            v = v * (1 - w / 100.0)
            notes.append(f"乾基→濕基：× (1 − {w:.2f}/100)")
    lo, hi, center = rr["low"], rr["high"], rr["center"]
    status = "範圍內" if lo <= v <= hi else ("偏低" if v < lo else "偏高")
    within = [f"{r['entry']} {'✓' if r['low'] <= v <= r['high'] else '✗'}" for r in rr["entries"]]
    ratio = v / center if center else _db_np.nan

    # (1) unit / factor errors: ratio ≈ 10^k, NaCl factor, forgotten dilution
    if status != "範圍內" and center and v > 0:
        cands = [(10, "10 倍（mg/kg vs mg/100 g；g/kg vs %；mg/L vs mg/100 mL）"),
                 (100, "100 倍（mg vs g/100 g？% vs mg/g？）"),
                 (1000, "1000 倍（mg vs g；μg vs mg）")]
        if nutrient == "sodium":
            cands.append((DB_NACL_FACTOR, "2.542 倍（NaCl 當量 vs Na，58.44/22.99）"))
        if dilution:
            cands.append((float(dilution), f"稀釋倍數 {dilution:g}（忘了乘或多乘了稀釋倍數？）"))
            if float(dilution) / 10 not in (1, 10, 100):
                cands.append((float(dilution) / 10, f"稀釋倍數/10 = {dilution / 10:g}（忘了 ×D 並 ÷10 換成每 100 g？）"))
        for f, why in cands:
            for direction, r in (("高", f), ("低", 1 / f)):
                if abs(_db_math.log10(ratio) - _db_math.log10(r)) < _db_math.log10(1.25):
                    flags.append(f"單位/換算錯誤？實測 ÷ 參考 ≈ {ratio:.3g}，約{direction} {why}")

    # (2) basis mismatch: dry-basis value reported as wet (or the reverse)
    wc = rr["water_center"]
    if wc is not None and wc >= 3 and center:
        ref_dry = center / (1 - wc / 100.0)
        if abs(v / ref_dry - 1) < 0.05 and abs(v / center - 1) > 0.08:
            flags.append(f"基準不一致？數值接近『乾基』參考值 {ref_dry:.3g}（= 濕基 {center:.3g} ÷ (1 − {wc:.1f}/100)）"
                         "；資料庫是『每 100 g 可食部分（濕基）』")
        if basis == "dry" and abs(float(value) / ref_dry - 1) > 0.08 and abs(float(value) / center - 1) < 0.05:
            flags.append("基準不一致？你標為乾基，但原始數值接近濕基參考值")

    # (3) method bias / method meaning
    m = (method or "").lower()
    tags = set(food.get("tags", []))
    hydrolysed = (("hydrol" in m or "水解" in m or "gottlieb" in m or "mojonnier" in m)
                  and not any(k in m for k in ("no hydrol", "without hydrol", "non-hydrol", "未水解")))
    if nutrient == "fat" and "soxhlet" in m and not hydrolysed and "dairy_powder" in tags:
        flags.append("方法偏差（預期偏低）：奶粉脂肪被蛋白質/乳糖包覆，直接 Soxhlet（未水解）萃取不完全；"
                     "乳品參考方法是 Röse–Gottlieb（氨–乙醇–乙醚，鹼水解）")
    if nutrient == "protein" and n_factor:
        for r in rr["entries"]:
            if r["n_factor"] and abs(r["n_factor"] - n_factor) > 1e-6:
                recalc = v / n_factor * r["n_factor"]
                flags.append(f"N 換算係數不同：你用 {n_factor:g}，{r['entry']} 用 {r['n_factor']:g} → "
                             f"重新換算 = {recalc:.2f} g/100 g")
                break
        else:
            if all(r["n_factor"] is None for r in rr["entries"]):
                notes.append("資料庫未提供 N 換算係數，無法重新換算")
    if nutrient == "sodium" and "ise" in m:
        notes.append("Na⁺ ISE 量的是離子活性（受離子強度/基質影響）；Na→NaCl ×2.542 假設所有鈉都來自 NaCl")
    if "phenol" in m:
        if nutrient == "sugars_total":
            flags.append("量的不是同一件事：酚–硫酸法是『以葡萄糖當量表示的總碳水化合物』（含寡醣、可水解多醣），"
                         "不是資料庫的『總糖』")
        elif nutrient == "carbohydrate":
            notes.append("酚–硫酸法（葡萄糖當量）≠ 差減法碳水化合物（100 − 其他四項，含膳食纖維與所有誤差）")
    if nutrient == "sugars_total" and "differ" in m:
        flags.append("量的不是同一件事：差減法碳水化合物 = 100 − (水分+灰分+蛋白質+脂肪)，包含所有碳水與所有誤差；"
                     "『總糖』只算單醣＋雙醣（奶粉主要是乳糖）")
    if nutrient == "carbohydrate" and "differ" in m:
        notes.append("差減法會繼承水分/灰分/蛋白質/脂肪的所有誤差：任何一項偏低 → 碳水化合物偏高")
    for r in rr["entries"]:
        if r["n"] == 0 and r["derivation"]:
            notes.append(f"{r['entry']} 的 {nutrient} 不是直接分析值（{r['derivation'][:40]}…）")
            break

    reminder = food.get("variability_note", "")
    if any(r["match"] == "closest" for r in rr["entries"]) or len(rr["entries"]) == 1:
        reminder = ("資料庫品項只是『最接近』或只有一筆，品牌/配方可能不同。" + reminder).strip()

    def _vals(db):
        return "; ".join(f"{r['id']}: {_db_fmt(r['value'])}" for r in rr["entries"] if r["db"] == db) or "—"

    return {"label": label if label is not None else "", "food_key": food_key, "name_zh": food["name_zh"],
            "nutrient": nutrient, "unit": f"{DB_UNITS[nutrient]}/100 g", "reported": float(value),
            "reported_unit": unit or "", "measured": round(v, 4), "sd": sd, "n": n,
            "USDA": _vals("USDA"), "TFND": _vals("TFND"),
            "ref_low": round(lo, 4), "ref_high": round(hi, 4), "ref_center": round(center, 4),
            "ratio_to_ref": round(ratio, 3), "status": status, "within_entries": ", ".join(within),
            "flags": " | ".join(flags), "notes": " | ".join(notes), "reminder": reminder}


def compare_table(ref, data, food_col="food_key", nutrient_col="nutrient", value_col="value",
                  unit_col="unit", **defaults):
    """Run check_measurement for every row of a tidy DataFrame.  Optional columns
    (used when present): label, sd, n, basis, method, n_factor, dilution, density, water."""
    out = []
    opt = ["label", "sd", "n", "basis", "method", "n_factor", "dilution", "density", "water"]
    for _, r in data.iterrows():
        kw = dict(defaults)
        for c in opt:
            if c in data.columns and _db_pd.notna(r[c]):
                kw[c] = r[c]
        out.append(check_measurement(ref, r[food_col], r[nutrient_col], r[value_col],
                                     unit=r[unit_col] if unit_col in data.columns else None, **kw))
    return _db_pd.DataFrame(out)


def show_flags(table):
    """Print status + flags + notes row by row (easier to read than a wide table)."""
    for _, r in table.iterrows():
        head = f"[{r['status']}] {r['label']} {r['name_zh']} {r['nutrient']}: {_db_fmt(r['measured'])} {r['unit']}"
        print(head + f"  (參考範圍 {r['ref_low']:.4g}–{r['ref_high']:.4g}；USDA {r['USDA']}；TFND {r['TFND']})")
        print(f"     逐筆是否涵蓋：{r['within_entries']}")
        if r["flags"]:
            for f in r["flags"].split(" | "):
                print("     ⚑ " + f)
        if r["notes"]:
            for f in r["notes"].split(" | "):
                print("     · " + f)


def plot_vs_reference(table, ncols=4, title=None):
    """Small multiples: grey bar = combined reference range, ^ = USDA values,
    s = TFND values, dot ± SD = your measurement (green in range, red outside)."""
    k = len(table)
    ncols = min(ncols, k)
    nrows = int(_db_np.ceil(k / ncols))
    fig, axes = _db_plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows), squeeze=False)
    for ax, (_, r) in zip(axes.flat, table.iterrows()):
        ax.bar(0, r["ref_high"] - r["ref_low"], bottom=r["ref_low"], width=0.6,
               color="lightgrey", edgecolor="grey", label="reference range")
        ys = [r["ref_low"], r["ref_high"], r["measured"]]
        for db, mk, dx, col in (("USDA", "^", -0.18, "tab:blue"), ("TFND", "s", 0.18, "tab:purple")):
            vals = [float(s.split(": ")[1]) for s in str(r[db]).split("; ") if ": " in s and s.split(": ")[1] != "NA"]
            if vals:
                ax.plot([dx] * len(vals), vals, mk, color=col, ms=7, label=db)
                ys += vals
        colr = "tab:green" if r["status"] == "範圍內" else "tab:red"
        err = r["sd"] if r["sd"] is not None and _db_pd.notna(r["sd"]) else None
        ax.errorbar(0, r["measured"], yerr=err, fmt="o", color=colr, ms=8, capsize=4, label="measured")
        if err:
            ys += [r["measured"] - err, r["measured"] + err]
        pad = 0.08 * (max(ys) - min(ys) or abs(max(ys)) or 1.0)
        ax.set_ylim(min(ys) - pad, max(ys) + pad)
        lab = str(r["label"])
        lab = lab if lab.isascii() else ""
        ax.set_title(f"{r['food_key']}\n{r['nutrient']} {lab}".strip(), fontsize=9)
        ax.set_ylabel(f"{r['nutrient']} ({r['unit']})", fontsize=8)
        ax.set_xticks([])
        ax.set_xlim(-0.6, 0.6)
    for ax in list(axes.flat)[k:]:
        ax.axis("off")
    leg = {}
    for ax in axes.flat:
        for h, l in zip(*ax.get_legend_handles_labels()):
            leg.setdefault(l, h)
    fig.legend(list(leg.values()), list(leg.keys()), loc="lower center", ncol=4, fontsize=8, frameon=False)
    if title:
        fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    _db_plt.show()
    return fig
# <<< DBCHECK-HELPERS-END

In [ ]:
db_ref = load_reference()          # 本機 data/ → GitHub → 內嵌子集（離線也能跑）

# 全脂奶粉在兩個資料庫中的數值、樣本數、換算係數與推導方式
db_rows = []
for db_nut in ["water", "ash", "protein", "fat", "carbohydrate", "sugars_total", "sodium"]:
    db_rr = reference_range(db_ref, "whole_milk_powder", db_nut)
    for e in db_rr["entries"]:
        db_rows.append({"nutrient": db_nut, "entry": e["entry"], "value": e["value"], "n": e["n"],
                        "sd": e["sd"], "N_factor": e["n_factor"],
                        "derivation": (e["derivation"] or "")[:28], "range_low": round(db_rr["low"], 2),
                        "range_high": round(db_rr["high"], 2)})
db_milk_ref = pd.DataFrame(db_rows)
db_milk_ref

### 9.1 全班 proximate 結果 vs 資料庫

用 5 組平均值的**中位數**代表全班（跟 z-score 的指定值同一個概念），SD 為組間標準差。脂肪標記方法 `soxhlet`（W6 直接 Soxhlet，未水解），蛋白質標記 `kjeldahl` 與本課使用的換算係數 6.38，碳水化合物標記 `difference`（差減法）。最後一列故意把「差減法碳水化合物」拿去跟資料庫的「總糖」比，看看會發生什麼事。

先猜猜看：哪幾項會跑出資料庫範圍？是往上還是往下？

In [ ]:
DB_W8_ITEMS = {  # proximate_df 欄位 -> (資料庫營養素, 方法, 你用的 N 換算係數)
    "moisture_%": ("water", None, None),
    "ash_%": ("ash", None, None),
    "protein_%": ("protein", "kjeldahl", 6.38),
    "fat_%": ("fat", "soxhlet", None),
    "carbohydrate_%": ("carbohydrate", "difference", None),
}
db_class_in = pd.DataFrame([
    dict(label="class", food_key="whole_milk_powder", nutrient=db_nut, unit="%",
         value=float(proximate_df[col].median()), sd=float(proximate_df[col].std(ddof=1)),
         n=len(proximate_df), method=db_m, n_factor=db_nf)
    for col, (db_nut, db_m, db_nf) in DB_W8_ITEMS.items()
])
db_class_in = pd.concat([db_class_in, pd.DataFrame([dict(
    label="class", food_key="whole_milk_powder", nutrient="sugars_total", unit="%",
    value=float(proximate_df["carbohydrate_%"].median()), method="difference")])], ignore_index=True)

db_class_tbl = compare_table(db_ref, db_class_in)
db_class_tbl[["nutrient", "measured", "sd", "USDA", "TFND", "ref_low", "ref_high", "status", "within_entries"]]

In [ ]:
show_flags(db_class_tbl)
plot_vs_reference(db_class_tbl, ncols=6, title="W8 blind whole milk powder: class median vs database")

### 9.2 脂肪：全班「直接 Soxhlet」一起偏低——整個方法的偏差

W8 彙整表的脂肪就是 **W6 的盲樣直接 Soxhlet（未水解）**三重複。把 5 組、全班平均，以及 Röse–Gottlieb 參考法數值（TA/參考實驗室提供）一起跟資料庫比：

In [ ]:
DB_RG_REFERENCE = 25.7   # Röse–Gottlieb 參考法（TA/參考實驗室提供；與 W6/NB06 同一份模擬資料）

fat_class_mean = float(np.mean([fat_mean[g] for g in GROUPS]))
fat_class_sd = float(np.std([fat_mean[g] for g in GROUPS], ddof=1))
db_fat_in = pd.DataFrame(
    [dict(label=g, food_key="whole_milk_powder", nutrient="fat", value=fat_mean[g], unit="%",
          method="soxhlet (no hydrolysis)") for g in GROUPS]
    + [dict(label="class mean", food_key="whole_milk_powder", nutrient="fat", value=fat_class_mean,
            sd=fat_class_sd, n=len(GROUPS), unit="%", method="soxhlet (no hydrolysis)"),
       dict(label="Rose-Gottlieb", food_key="whole_milk_powder", nutrient="fat", value=DB_RG_REFERENCE,
            unit="%", method="rose_gottlieb")]
)
db_fat_tbl = compare_table(db_ref, db_fat_in)
print(f"全班直接 Soxhlet 平均 = {fat_class_mean:.2f} ± {fat_class_sd:.2f}%（組間 SD）；"
      f"Röse–Gottlieb = {DB_RG_REFERENCE:.1f}%；bias = {fat_class_mean - DB_RG_REFERENCE:+.2f} 個百分點")
db_fat_tbl[["label", "measured", "sd", "ref_low", "ref_high", "status", "within_entries"]]

In [ ]:
show_flags(db_fat_tbl[db_fat_tbl["label"].isin(["class mean", "Rose-Gottlieb"])])
plot_vs_reference(db_fat_tbl[db_fat_tbl["label"].isin(["class mean", "Rose-Gottlieb"])],
                  ncols=2, title="Fat: class direct Soxhlet vs Rose-Gottlieb vs database")

5 組**彼此非常接近**（組間 SD 不到 0.5 個百分點），卻**一起**落在資料庫範圍之下，而 Röse–Gottlieb 參考法落在範圍內——這是**方法本身的系統偏差**（脂肪被蛋白質/乳糖包覆，未水解就萃取不完全），不是哪一組做錯。
因為 z-score 的指定值是全班中位數，全班一起偏低時，每一組的脂肪 z-score 看起來都很「正常」（見問題 7）；只有拿外部參考（資料庫、參考方法）來比，才抓得到這種偏差。

### 9.3 蛋白質：G4 在資料庫範圍內，但 z-score 不合格

蛋白質用 Kjeldahl 測總氮再乘上換算係數。**USDA 的全脂奶粉用 6.38**（乳品係數，見上表 `N_factor`）；TFND 開放資料沒有註明係數。本課 W5 也是用 6.38，所以可以直接比。

In [ ]:
db_prot_in = pd.DataFrame([dict(label=g, food_key="whole_milk_powder", nutrient="protein", value=prot_mean[g],
                                unit="%", method="kjeldahl", n_factor=6.38) for g in GROUPS])
db_prot_tbl = compare_table(db_ref, db_prot_in)
db_prot_tbl["z"] = [z_score[g] for g in GROUPS]
db_prot_tbl["z_category"] = [classify_z(z_score[g]) for g in GROUPS]
db_prot_tbl[["label", "measured", "ref_low", "ref_high", "status", "within_entries", "z", "z_category"]]

In [ ]:
# 換算係數的影響：同樣的含氮量，若誤用一般係數 6.25，會得到較低的蛋白質%
db_N_pct = protein_reference_value / 6.38                 # 全班中位數換回含氮量 (%N)
db_p625 = check_measurement(db_ref, "whole_milk_powder", "protein", db_N_pct * 6.25, unit="%",
                            method="kjeldahl", n_factor=6.25, label="class N x 6.25")
print(f"含氮量 = {db_N_pct:.3f} %N；x6.38 = {protein_reference_value:.2f} %；x6.25 = {db_N_pct * 6.25:.2f} %")
print("旗標：", db_p625["flags"])

### 9.4 碳水化合物（差減法）與總糖：不是同一件事

差減法碳水化合物＝100 −（水分＋灰分＋蛋白質＋脂肪），**任何一項偏低都會讓碳水偏高**。先看全班：碳水化合物被資料庫標成什麼？再把「全班 − 資料庫」的差拆回五個成分，找出是哪一項把碳水墊高的。

In [ ]:
show_flags(db_class_tbl[db_class_tbl["nutrient"] == "carbohydrate"])

# 全班中位數 − 資料庫中心值（USDA/TFND 各筆平均），單位：百分點
db_trace = pd.DataFrame([
    dict(component=col, class_median=round(float(proximate_df[col].median()), 3),
         db_center=round(reference_range(db_ref, "whole_milk_powder", db_nut)["center"], 3))
    for col, (db_nut, _m, _nf) in DB_W8_ITEMS.items()
])
db_trace["class_minus_db"] = (db_trace["class_median"] - db_trace["db_center"]).round(3)
db_trace

In [ ]:
# 如果脂肪用 Röse–Gottlieb 參考值（其他三項不變），碳水化合物會回到資料庫範圍內嗎？
db_carb_rg = {g: 100.0 - (moist_mean[g] + ash_mean[g] + prot_mean[g] + DB_RG_REFERENCE) for g in GROUPS}
db_carb_rg_tbl = compare_table(db_ref, pd.DataFrame(
    [dict(label=g, food_key="whole_milk_powder", nutrient="carbohydrate", value=db_carb_rg[g], unit="%",
          method="difference") for g in GROUPS]))
pd.DataFrame({"group": GROUPS,
              "carb_soxhlet_%": [carb_mean[g] for g in GROUPS],
              "carb_if_fat_RG_%": [round(db_carb_rg[g], 3) for g in GROUPS],
              "status_if_fat_RG": db_carb_rg_tbl["status"].tolist()})

「全班 − 資料庫」這一欄裡，**脂肪約 −6、碳水約 +6，其他三項都在 ±0.5 以內**——碳水化合物偏高幾乎完全是脂肪低估「傳」過來的（誤差傳遞）。把脂肪換成 Röse–Gottlieb 參考值之後，5 組的碳水全部回到資料庫範圍內。
所以看到差減法碳水偏高時，**先回頭檢查其他四項**，而不是去懷疑碳水化合物本身。

下表則是「組別層級」：每組相對**全班中位數**的偏離。全班共同的脂肪偏差在這裡會被中位數抵銷掉，看得到的是組別特有的問題（例如 G4 蛋白質偏低 → G4 碳水最高）。

In [ ]:
db_carb_in = pd.DataFrame([dict(label=g, food_key="whole_milk_powder", nutrient="carbohydrate",
                                value=carb_mean[g], unit="%", method="difference") for g in GROUPS])
db_carb_tbl = compare_table(db_ref, db_carb_in)
db_comp = proximate_df.set_index("group")[["moisture_%", "ash_%", "protein_%", "fat_%", "carbohydrate_%"]]
db_comp_dev = (db_comp - db_comp.median()).round(3)      # 各組相對全班中位數的偏離 (百分點)
db_comp_dev["db_status_carb"] = db_carb_tbl.set_index("label")["status"]
db_comp_dev

In [ ]:
# 資料庫裡的「總糖」是怎麼來的？（看 n 與 derivation）
_ = reference_range(db_ref, "whole_milk_powder", "sugars_total", verbose=True)
for e in db_food(db_ref, "whole_milk_powder")["entries"]:
    v = e["nutrients"].get("sugars_total", {})
    print(f"  {e['db']} {e['id']}: sugars_total = {v.get('value')}  n = {v.get('n')}  derivation = {v.get('derivation', '—')}")

- USDA 170876 的「總糖」38.42 g 與差減法碳水化合物**完全相同**，而且 n = 0、推導代碼 NR（由其他營養素直接帶入）——它是**假設**「奶粉的碳水幾乎全是乳糖」而來，不是另外分析的值。
- TFND 則是實際分析糖質（乳糖 36.4 g、n = 1），總碳水化合物 37.0 g——兩者接近但不相等。
- 所以「差減法碳水化合物」「總糖（乳糖）」與 W7 的「酚–硫酸法總碳水（葡萄糖當量）」是**三個不同的量**，比對前要先確認資料庫欄位的定義。

### 9.5 「我這組有問題」vs「全班／方法有問題」

| 線索 | 我這組有問題（group-specific） | 全班／方法有問題（class-wide / method bias） |
|---|---|---|
| 跟全班比（z-score、Grubbs） | 只有我這組偏離 | 各組彼此接近，z-score 看起來都「滿意」 |
| 跟資料庫或參考方法比 | 常常仍在資料庫範圍內（範圍反映產品間差異，很寬） | 全班一起偏向同一側，甚至一起落在範圍外 |
| 下一步 | 檢查本組的計算、試劑濃度、操作，重做 | 換方法或加前處理（例如水解），用參考物質/參考方法校正 |

兩種檢查**互補**：z-score 的指定值來自全班中位數，所以全班一起偏掉時它看不出來；資料庫範圍很寬，所以單一組別的小錯它也看不出來。

### （選用）用 R 做同樣的比對

R 使用者可以用 `jsonlite` 讀同一份 `reference_foods.json`，照同樣規則算範圍：

```r
# install.packages("jsonlite")   # 若尚未安裝
library(jsonlite)
path <- "data/reference_foods.json"
if (!file.exists(path))
  path <- "https://raw.githubusercontent.com/Tai-ShengYeh/food-analysis-lab/main/data/reference_foods.json"
ref <- fromJSON(path, simplifyVector = FALSE)

tol  <- c(water = .10, protein = .10, fat = .15, ash = .15, carbohydrate = .10, sugars_total = .15, sodium = .15)
absf <- c(water = .3,  protein = .3,  fat = .3,  ash = .05, carbohydrate = 1,   sugars_total = .5,  sodium = 5)
nz <- function(x) if (is.null(x)) NA else x

ref_entries <- function(ref, food_key, nutrient) {
  food <- Filter(function(f) f$key == food_key, ref$foods)[[1]]
  rows <- lapply(food$entries, function(e) {
    v <- e$nutrients[[nutrient]]
    if (is.null(v)) return(NULL)
    half <- max(tol[[nutrient]] * abs(v$value), absf[[nutrient]])
    lo <- v$value - half; hi <- v$value + half
    if (!is.null(v$min) && !is.null(v$max)) { lo <- min(lo, v$min); hi <- max(hi, v$max) }
    else if (!is.null(v$sd) && nz(v$n) >= 2) { lo <- min(lo, v$value - 2 * v$sd); hi <- max(hi, v$value + 2 * v$sd) }
    data.frame(db = e$db, id = as.character(e$id), name = e$name, value = v$value,
               n = nz(v$n), low = max(lo, 0), high = hi)
  })
  do.call(rbind, rows)
}

tab <- ref_entries(ref, "whole_milk_powder", "fat")
print(tab)
measured <- 21.28                      # 例：W6 全班直接 Soxhlet 平均（g/100 g）
rng <- c(min(tab$low), max(tab$high))
status <- if (measured < rng[1]) "low" else if (measured > rng[2]) "high" else "within"
cat(sprintf("reference range %.2f-%.2f g/100 g -> %s\n", rng[1], rng[2], status))
```

> 另外，CRAN 套件 **NutrienTrackeR** 內建數個國家的食物成分資料庫（包含 USDA），可用於飲食營養計算；但它**沒有**台灣的 TFND，所以本課的比對仍以 `reference_foods.json`（同時收 USDA 與 TFND）為準。

## 10. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果 G3 水分的第 3 重複值不是 3.30%，而是誤差更大的 3.60%，重新計算 Grubbs 的 $G$ 值。這樣是否仍然「測不出來」（$G < G_{crit}$）？

In [ ]:
# TODO: 請在這裡作答

### 問題 2

在「班級層級 Grubbs」（5 組蛋白質單次測定值）中，除了被標記離群的那一組之外，找出 $G$ 值第二大的組別，並查它的 z-score 落在哪個等級（satisfactory / questionable / unsatisfactory）。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

比較灰分 ANOVA 算出來的 $s_r$（重複性）與 $s_L$（組間）何者較大？這代表什麼意義？如果某次改用更嚴謹的操作，使得 $s_L$ 幾乎降到 0，$s_R$ 會變得跟什麼很接近？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

蛋白質不確定度預算中，哪一個分量貢獻最大？如果能把「滴定劑濃度」的相對標準不確定度從 0.30% 降到 0.10%（重新標定、用更精確的方法），合併不確定度 $u_c$（相對）會變成多少？影響大不大？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

如果巧克力水分的規格從 $\le2\%$ 放寬成 $\le2.5\%$，測得結果仍是 $2.30\% \pm 0.10\%$，用 guard band 規則重新判定，結果會不會跟原規格（$\le2\%$，判定「不放行」）不一樣？

In [ ]:
# TODO: 請在這裡作答

### 問題 6：換算係數能不能解釋 G4？

假設 G4 其實是把含氮量乘上一般係數 **6.25**（而不是 6.38）才得到 24.47%。

1. 用 `check_measurement(..., n_factor=6.25)` 把 G4 重新換算成 6.38 的蛋白質%。
2. 用第 5 節同樣的指定值與 $\sigma_{pt}$ 重新計算 G4 的 z-score，分級會變成什麼？
3. 「用錯換算係數」能不能完全解釋 G4 的偏低？還需要找什麼原因？

In [ ]:
# TODO: 請在這裡作答

### 問題 7：全班一起偏掉時，z-score 看得出來嗎？

本班 5 組的盲樣脂肪（`fat_mean`，W6 直接 Soxhlet）彼此很接近。

1. 用這 5 組脂肪平均的中位數當指定值、$\sigma_{pt}=0.40\%$，算出各組脂肪的 z-score，分級如何？
2. 用 `compare_table()` 跟資料庫比（方法填 `"soxhlet"`），結果與旗標是什麼？
3. 這是「某一組有問題」還是「全班／方法有問題」？只看 z-score 會得到什麼錯誤結論？

In [ ]:
# TODO: 請在這裡作答

## 11. 匯出結果 (Export)

把彙整後的 proximate 表、Grubbs/ANOVA/z-score/放行判定結果存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載（用 try/except 保護，確保兩種環境都能跑）。

In [ ]:
output_path = "nb08_proximate_results.csv"
proximate_df.merge(z_df[["group", "z", "category"]], on="group").to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

release_output_path = "nb08_release_decisions.csv"
release_cases.to_csv(release_output_path, index=False)
print(f"已儲存：{release_output_path}")

try:
    from google.colab import files
    files.download(output_path)
    files.download(release_output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")